# 01 — Data Exploration

This notebook lets you visually explore your Forex data before training.
Run cells one by one (Shift+Enter).

In [ ]:
import os, sys
sys.path.insert(0, '../src')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import yaml

# Load config
with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

pair      = cfg['pair'].replace('=X','').lower()
timeframe = cfg['timeframe']
print(f'Pair: {pair} | Timeframe: {timeframe}')

## 1. Load Raw Data

Make sure you've run `python src/data_loader.py` first.

In [ ]:
raw_file = f'../data/raw/{pair}_{timeframe}.csv'
df = pd.read_csv(raw_file, parse_dates=['datetime'])
print(f'Shape: {df.shape}')
print(f'Date range: {df["datetime"].min()} → {df["datetime"].max()}')
df.head()

## 2. Price Chart

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ax1.plot(df['datetime'], df['close'], color='#4a90d9', linewidth=0.8)
ax1.set_title(f'{pair.upper()} Close Price ({timeframe} candles)')
ax1.set_ylabel('Price')

ax2.bar(df['datetime'], df['volume'], color='#9b59b6', alpha=0.6, width=0.03)
ax2.set_ylabel('Volume')
ax2.set_xlabel('Date')

plt.tight_layout()
plt.show()

## 3. Load Feature-Engineered Data

Make sure you've run `python src/feature_engineer.py` first.

In [ ]:
feat_file = f'../data/processed/{pair}_{timeframe}_features.csv'
df_feat = pd.read_csv(feat_file, parse_dates=['datetime'])
print(f'Shape: {df_feat.shape}')
print('Columns:', list(df_feat.columns))

## 4. Technical Indicators Chart

In [ ]:
# Plot last 200 candles for clarity
plot_df = df_feat.tail(200)

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

# Price + MAs
axes[0].plot(plot_df['datetime'], plot_df['close'],    label='Close',    linewidth=0.8)
axes[0].plot(plot_df['datetime'], plot_df['ma_short'], label='MA Short', linewidth=1, linestyle='--')
axes[0].plot(plot_df['datetime'], plot_df['ma_long'],  label='MA Long',  linewidth=1, linestyle='-.')
axes[0].fill_between(plot_df['datetime'], plot_df['bb_upper'], plot_df['bb_lower'], alpha=0.1, color='gray')
axes[0].set_title('Price + Moving Averages + Bollinger Bands')
axes[0].legend(fontsize=8)

# RSI
axes[1].plot(plot_df['datetime'], plot_df['rsi'], color='#e67e22', linewidth=1)
axes[1].axhline(70, color='red',   linestyle='--', linewidth=0.8, label='Overbought')
axes[1].axhline(30, color='green', linestyle='--', linewidth=0.8, label='Oversold')
axes[1].axhline(50, color='gray',  linestyle=':',  linewidth=0.6)
axes[1].set_ylabel('RSI')
axes[1].set_ylim(0, 100)
axes[1].legend(fontsize=8)

# MACD
axes[2].plot(plot_df['datetime'], plot_df['macd'],     label='MACD',   linewidth=1)
axes[2].plot(plot_df['datetime'], plot_df['macd_sig'], label='Signal', linewidth=1, linestyle='--')
axes[2].bar(plot_df['datetime'], plot_df['macd_diff'], color='gray', alpha=0.4, width=0.03)
axes[2].axhline(0, color='black', linewidth=0.5)
axes[2].set_ylabel('MACD')
axes[2].legend(fontsize=8)

# ATR
axes[3].plot(plot_df['datetime'], plot_df['atr_pct'] * 100, color='#9b59b6', linewidth=1)
axes[3].set_ylabel('ATR %')
axes[3].set_xlabel('Date')

plt.tight_layout()
plt.show()

## 5. Check Label Distribution

Make sure you've run `python src/labeller.py` first.

In [ ]:
train_file = f'../data/labelled/{pair}_{timeframe}_train.csv'
df_train = pd.read_csv(train_file)

up_pct = df_train['target'].mean() * 100
print(f'UP: {up_pct:.1f}%   DOWN: {100-up_pct:.1f}%')
print('Note: 45-55% split is healthy. Strongly imbalanced data may need resampling.')

df_train['target'].value_counts().plot(kind='bar', color=['#e74c3c','#2ecc71'],
                                        title='Label distribution (0=DOWN, 1=UP)')
plt.xticks([0, 1], ['DOWN', 'UP'], rotation=0)
plt.show()

## 6. Feature Correlations

Check which features are most correlated with the target.

In [ ]:
import seaborn as sns

non_feat = ['datetime','target','open','high','low','close','volume']
feat_cols = [c for c in df_train.columns if c not in non_feat]

corr = df_train[feat_cols + ['target']].corr()['target'].drop('target').sort_values()

fig, ax = plt.subplots(figsize=(6, 10))
corr.plot(kind='barh', ax=ax, color=['#e74c3c' if v < 0 else '#2ecc71' for v in corr])
ax.axvline(0, color='black', linewidth=0.5)
ax.set_title('Feature correlation with UP/DOWN target')
ax.set_xlabel('Pearson correlation')
plt.tight_layout()
plt.show()